In [2]:
import json
PDF_PATH = '../data/wheat_traits_2013.pdf'
MD_PATH = '../data/wheat_traits_2013.md'


In [3]:
# import pymupdf4llm
# md_text = pymupdf4llm.to_markdown(PDF_PATH)
md_txt = open(MD_PATH, 'r').read()


In [4]:
md_text_page = md_txt[98000:102000]
print(md_text_page)

agnostic' association of _Rht8c_ and _Xgwm261192_ applied in many Strampelli derivatives and European wheats, there was no association between reduced height and this allele in Norin 10 and its derivatives {10512}. The pedigrees of a number of Chinese wheats postulated to have _Rht8c_ on the basis of the marker trace to Italian sources {10515}. 

_**Rht8d**_ .   Associated with a 201-bp fragment of WMS261 {9962}. **v:** Pliska{9962}; Courtot{9962}. 

_**Rht8e**_ .   Associated with a 210-bp fragment of WMS261 {9962}. **v:** Chino{9962}; Klein Esterello{9962}; Klein 157{9962}. 

   - _**Rht8f**_ .   Associated with a 215-bp fragment of WMS261 {9962}. **v:** Klein 49{9962}. _**Rht8g**_ .   Associated with a 196-bp fragment of WMS261 [{0243}]. **v:** Mirleben{0243}. _**Rht8h**_ .   Associated with a 206-bp fragment of WMS261 [{0243}]. **v:** Weihenstephan M1{0243}. 

- _**Rht9**_ .   7BS{772,1601}.5AL{10249}. **v:** Acciao{718}; Forlani{718}; Mercia 12{10249}. **s:** Cappelle-Desprez[*] /

In [18]:
from pyparsing import (Regex, Group, Suppress, Literal, ParserElement,
                       alphanums, nums, Word, SkipTo, ZeroOrMore, Optional)

ParserElement.DEFAULT_WHITE_CHARS = ' \t'

# Trait header: **11.1. Name** with optional leading ##
trait_pat = (
    Suppress(Regex(r'#{0,2}\s*\*\*')) +
    Regex(r'(?:\d+\.)+')('index') +
    Regex(r'[^\n*]+')('name') +
    Suppress(Literal('**'))
)

# Gene id: _**GeneName**_ {optional_citation}.
gene_pat = Regex(r'_\*\*[^*\n]+\*\*_')
gene_pat.add_parse_action(lambda t: t[0][3:-3])
citation = Regex(r'\{[^}]+\}')('citation')
gene_id = gene_pat('gene') + Optional(citation) + Suppress(Literal('.'))

# Annotation: **label:** value until next period
BOLD = Literal('**')
gene_annot = (
    Suppress(BOLD) +
    Word(alphanums)('label') +
    Suppress(Literal(':')) +
    Suppress(BOLD) +
    SkipTo('.')('value') +
    Suppress(Literal('.'))
)

alt_annot = (Suppress(Literal('[')) + SkipTo(']')('alt') + Suppress(Literal(']')))
chromosome_annot = (Regex(r'\d[A-Z]{1,2}')('chrom') +
                    Optional(Regex(r'\[.*\]')) + Optional(citation)
                    + Suppress(Literal('.'))
                    )

# Lookahead anchor: where a new entry begins
next_anchor = trait_pat | gene_id

# Free-text description: everything before the first **label:** annotation or next entry.
# Note: chromosome annotations (e.g. "7BS{772}.") are left as part of this free text.
gene_descr = SkipTo(gene_annot | next_anchor)('desc')
gene_entry = (gene_id + 
                Optional(alt_annot) +
                Optional(chromosome_annot) + 
                gene_descr + 
                ZeroOrMore(Group(gene_annot))('annots'))

In [20]:
s = "- _**Rht13**_ {718}.   7BS. **v:** Magnif 41M1 CI 17689{718}. **ma:** Associated with 9.9 _Xwms5777B_ {10249}."
parsed_s = (Suppress(SkipTo(gene_id)) + gene_entry).parse_string(s)

In [21]:
print(s)
print(parsed_s.dump(include_list=False))

- _**Rht13**_ {718}.   7BS. **v:** Magnif 41M1 CI 17689{718}. **ma:** Associated with 9.9 _Xwms5777B_ {10249}.

- annots: 
  [0]:
    
    - label: 'v'
    - value: 'Magnif 41M1 CI 17689{718}'
  [1]:
    
    - label: 'ma'
    - value: 'Associated with 9'
- chrom: '7BS'
- citation: '{718}'
- desc: ''
- gene: 'Rht13'
[0]:
  Rht13
[1]:
  {718}
[2]:
  7BS
[3]:
  
[4]:
  
  - label: 'v'
  - value: 'Magnif 41M1 CI 17689{718}'
[5]:
  
  - label: 'ma'
  - value: 'Associated with 9'


In [44]:
parsed_s

ParseResults(['Rht13', '{718}', '   7BS.', ParseResults(['v', 'Magnif 41M1 CI 17689{718}'], {'label': 'v', 'value': 'Magnif 41M1 CI 17689{718}'}), ParseResults(['ma', 'Associated with _Xwms5777B_ {10249}'], {'label': 'ma', 'value': 'Associated with _Xwms5777B_ {10249}'})], {'gene': 'Rht13', 'citation': '{718}', 'descr': '   7BS.', 'annots': [{'label': 'v', 'value': 'Magnif 41M1 CI 17689{718}'}, {'label': 'ma', 'value': 'Associated with _Xwms5777B_ {10249}'}]})

In [19]:
entry = Group(
    (Group(trait_pat)('trait') | Group(gene_entry)('gene_entry')) +
    SkipTo(next_anchor)('description')
)

grammar = Suppress(SkipTo(next_anchor)) + ZeroOrMore(entry)


def segment(text):
    """
    Returns a flat list of dicts with 'type' in:
      'trait'            — numbered section header
      'trait_description'— unstructured text following a trait header
      'gene'             — structured gene entry: gene, citation, descr, annots
      'gene_description' — leftover text after gene_entry that wasn't parsed (rare)
    """
    segments = []
    for e in grammar.parse_string(text):
        desc = e.description.strip()
        if 'trait' in e:
            h = e.trait
            segments.append({'type': 'trait', 'index': h.index.strip(), 'name': h.name.strip()})
            key = 'trait'
        else:
            g = e.gene_entry
            seg = {
                'type':     'gene',
                'gene':     g.gene,
                'citation': g.citation or None,
                'chromosome': g.chrom or None,
                'descr':    g.descr.strip(),
                'annots':   {a.label: a.value.strip() for a in g.annots},
            }
            segments.append(seg)
            key = 'gene'
        if desc:
            segments.append({'type': f'{key}_description', 'text': desc})
    return segments


# Apply to the snippet
for seg in segment(md_text_page):
    t = seg['type']
    if t == 'trait':
        print(f"\n=== [{seg['index']}] {seg['name']}")
    elif t == 'gene':
        cit = f" {seg['citation']}" if seg['citation'] else ''
        print(f"  GENE: {seg['gene']}{cit}")
        if seg['descr']:
            print(f"    descr: {seg['descr'][:80]}")
        for label, val in seg['annots'].items():
            print(f"    {label}: {val[:80]}")
    else:
        print(f"  [{t}]: {seg['text'][:100]}")

  GENE: Rht8d
    v: Pliska{9962}; Courtot{9962}
  GENE: Rht8e
    v: Chino{9962}; Klein Esterello{9962}; Klein 157{9962}
  [gene_description]: -
  GENE: Rht8f
    v: Klein 49{9962}
  GENE: Rht8g
    v: Mirleben{0243}
  GENE: Rht8h
    v: Weihenstephan M1{0243}
  [gene_description]: -
  GENE: Rht9 {772,1601}
    v: Acciao{718}; Forlani{718}; Mercia 12{10249}
    s: Cappelle-Desprez[*] /Mara 5BS-7BS{1601}
    v2: Akakomugi _Rht8_ {1601}; Mara _Rht8_ {1601}
    ma: Close linkage with _Xwmc410-4A_ {10249}
  GENE: Rht11 {718}
    v: Karlik 1{718}
  GENE: Rht12 {718}
    bin: 5AL-23, based on co-segregation with 

_B1_ {1606}
    v: Karcagi 522M7K{721}
    ma: _Rht12_ is located distally on 5AL 

cosegregating with _B1_ and closely linked 
  [gene_description]: 4 cM - _Rht12_ {726}. 

_Rht12_ delayed ear emergence by 6 days{1606}. 

-
  GENE: Rht13 {718}
    v: Magnif 41M1 CI 17689{718}
    ma: Associated with _Xwms5777B_ {10249}
  [gene_description]: -
  GENE: Rht14 {718}
    v: Cp B 132 
